<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A;
             padding-bottom:.15em; margin-bottom:.5em; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size:.62em; border-radius:6px; border:1px solid #E2E6EE; box-shadow:none; }
.reveal pre code { max-height:420px; }
.reveal ul li, .reveal ol li { margin-bottom:.3em; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E;
        padding:.6em 1em; border-radius:0 6px 6px 0; margin:.5em 0; }
</style>

## Simulación Monte Carlo
### Modelado de Sistemas bajo Incertidumbre · Semanas 1–2
---
Departamento de Ingeniería Industrial · Universidad de los Andes

## Agenda
1. Contexto: problema de ingeniería con incertidumbre
2. Modelo matemático y función de transferencia
3. Paso 1 — Distribuciones de las variables de entrada
4. Paso 2 — Generación de muestras aleatorias
5. Pasos 3 y 4 — Evaluación y análisis de resultados
6. Métricas estadísticas: media, IC 95%, P(pérdida)
7. Métricas de riesgo: VaR y CVaR
8. Análisis de sensibilidad
9. Convergencia: ¿cuántas réplicas necesito?

## Contexto: Lanzamiento de Producto bajo Incertidumbre

<div class="defn">

**Situación:** Una empresa manufacturera evalúa lanzar una nueva línea de empaques sostenibles. Los costos de producción y la demanda del mercado son **inciertos**. ¿El proyecto es rentable?

</div>

| Variable | Descripción | Tipo |
|---|---|---|
| Precio de venta $P$ | $\$90{,}000$ COP/unidad | **Fijo** (precio de contrato) |
| Costo variable $C_v$ | Depende de insumos y mano de obra | **Incierto** |
| Demanda anual $D$ | Depende del comportamiento del mercado | **Incierta** |
| Costos fijos $C_f$ | Nómina base, arriendo, equipo | **Fijo** |

**Pregunta central:** ¿Cuál es la *distribución* de la utilidad anual? ¿Cuál es el riesgo de pérdida?

## Modelo Matemático

**Función de transferencia** (descripción del sistema):
$$U = \underbrace{(P - C_v)}_{\text{margen unitario}} \times D \;-\; C_f$$

**Variables estocásticas** (datos del estudio de mercado y producción):
$$C_v \sim U(35{,}000,\ 55{,}000) \quad \text{COP/unidad}$$
$$D \sim N(2{,}500,\ 500) \quad \text{unidades/año}$$

**Parámetros fijos:** $P = \$90{,}000$ COP/unidad, $\quad C_f = \$60{,}000{,}000$ COP/año

<div class="nota">

**Análisis determinista (ingenuo):** con solo los valores medios, $U = (90k - 45k) \times 2500 - 60M = \$52.5\text{M}$. Esto **ignora por completo la variabilidad y el riesgo**. Monte Carlo nos da la distribución completa de $U$.

</div>

## Paso 1 — Distribuciones de las Variables de Entrada

**¿Cómo elegimos la distribución?**

| Variable | Distribución elegida | Justificación |
|---|---|---|
| $C_v \sim U(35k, 55k)$ | Uniforme | Solo conocemos el rango; sin preferencia por ningún valor |
| $D \sim N(2500, 500)$ | Normal | Suma de muchos factores de mercado → TLC |

**Momentos clave:**
- $U(a,b)$: $\mu = (a+b)/2 = 45k$, $\quad \sigma = (b-a)/\sqrt{12} \approx 5{,}774$
- $N(\mu, \sigma)$: $\mu = 2500$, $\quad \sigma = 500$

El código siguiente grafica las distribuciones de ambas variables de entrada con su PDF teórica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(2024)

# --- Parámetros del modelo ---
P      = 90_000          # Precio de venta fijo (COP/unidad)
Cf     = 60_000_000      # Costos fijos anuales (COP)
cv_min, cv_max = 35_000, 55_000   # Costo variable: U(35k, 55k)
d_mu, d_sigma  = 2_500, 500       # Demanda: N(2500, 500)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Distribución de Cv ~ U(35k, 55k)
cv_samp = np.random.uniform(cv_min, cv_max, 10_000)
axes[0].hist(cv_samp/1_000, bins=50, color='#1565C0', edgecolor='white', alpha=0.85, density=True)
x_cv = np.linspace(cv_min - 1000, cv_max + 1000, 300)
axes[0].plot(x_cv/1_000, stats.uniform.pdf(x_cv, cv_min, cv_max - cv_min),
             '#C8961E', lw=2.5, label='PDF teórica')
axes[0].axvline(np.mean(cv_samp)/1_000, color='#C8961E', lw=2, ls='--',
                label=f'Media = ${np.mean(cv_samp)/1_000:.0f}k')
axes[0].set_title('$C_v \\sim U(35k,\\ 55k)$  COP/unidad', fontsize=10)
axes[0].set_xlabel('Costo variable (miles COP)'); axes[0].set_yticks([])
axes[0].legend(fontsize=9)

# Distribución de D ~ N(2500, 500)
d_samp = np.random.normal(d_mu, d_sigma, 10_000)
axes[1].hist(d_samp, bins=50, color='#1A7A9A', edgecolor='white', alpha=0.85, density=True)
x_d = np.linspace(d_mu - 4*d_sigma, d_mu + 4*d_sigma, 300)
axes[1].plot(x_d, stats.norm.pdf(x_d, d_mu, d_sigma), '#C8961E', lw=2.5, label='PDF teórica')
axes[1].axvline(d_mu, color='#C8961E', lw=2, ls='--', label=f'Media = {d_mu}')
axes[1].set_title('$D \\sim N(2500,\\ 500)$  unidades/año', fontsize=10)
axes[1].set_xlabel('Demanda (unidades/año)'); axes[1].set_yticks([])
axes[1].legend(fontsize=9)

plt.suptitle('Paso 1 — Distribuciones de las Variables de Entrada',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## Paso 2 — Generar Muestras Aleatorias

Para cada réplica $i = 1, 2, \ldots, n$ se extrae **un valor independiente** de cada distribución:

$$c_{v,i} \sim U(35{,}000,\ 55{,}000) \qquad d_i \sim N(2{,}500,\ 500)$$

Cada par $(c_{v,i},\ d_i)$ representa **un escenario posible** del negocio.

```python
# En Python, generar n escenarios simultáneamente es simplemente:
Cv = np.random.uniform(35_000, 55_000, n)   # vector de n costos variables
D  = np.random.normal(2_500, 500, n)         # vector de n demandas
```

Las primeras 8 réplicas ilustran la variabilidad de los escenarios:

In [ ]:
import numpy as np
np.random.seed(2024)

P, Cf = 90_000, 60_000_000
cv_min, cv_max = 35_000, 55_000
d_mu, d_sigma  = 2_500, 500

print(f"{'Rep':>4}  {'Cv (COP/u)':>13}  {'D (u/año)':>10}  {'Margen/u':>11}  {'Utilidad':>14}  {'Resultado'}")
print("─" * 72)
for i in range(8):
    cv = np.random.uniform(cv_min, cv_max)
    d  = np.random.normal(d_mu, d_sigma)
    u  = (P - cv) * d - Cf
    res = "✓ Ganancia" if u > 0 else "✗ Pérdida"
    print(f"  {i+1:>2}  ${cv:>10,.0f}   {d:>9,.0f}   ${P-cv:>8,.0f}   ${u/1e6:>9.2f} M  {res}")
print("─" * 72)
print(f"  ...   (continuamos hasta n = 10,000 réplicas)")

## Pasos 3 y 4 — Evaluar el Modelo y Analizar Resultados

**Paso 3 — Función de transferencia** (aplicada a cada réplica):
$$u_i = (P - c_{v,i}) \times d_i - C_f$$

**Paso 4 — Analizar** la distribución de $\{u_1, u_2, \ldots, u_n\}$:

<div class="nota">

La distribución de $U$ **no es Normal**, aunque $D$ lo sea. Esto ocurre porque $U$ contiene el **producto** de dos variables aleatorias: $(P - C_v) \times D$. Monte Carlo no necesita conocer esta distribución analíticamente — la *construye* empíricamente.

</div>

Con $n = 10{,}000$ réplicas graficamos el histograma y la función de distribución acumulada (FDA):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(2024)

P, Cf = 90_000, 60_000_000
cv_min, cv_max = 35_000, 55_000
d_mu, d_sigma  = 2_500, 500
N = 10_000

Cv = np.random.uniform(cv_min, cv_max, N)
D  = np.random.normal(d_mu, d_sigma, N)
U  = (P - Cv) * D - Cf          # función de transferencia: utilidad de cada escenario

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Histograma ---
axes[0].hist(U/1e6, bins=70, color='#1A7A9A', edgecolor='white', alpha=0.85, label='Ganancia')
axes[0].hist(U[U < 0]/1e6, bins=20, color='#C62828', edgecolor='white', alpha=0.7, label='Pérdida')
axes[0].axvline(0, color='#C62828', lw=2, ls='--', label='U = 0  (equilibrio)')
axes[0].axvline(np.mean(U)/1e6, color='#C8961E', lw=2,
                label=f'$\\hat{{E}}[U]$ = ${np.mean(U)/1e6:.1f}$ M')
axes[0].set_xlabel('Utilidad anual (M$ COP)'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title(f'Distribución de Utilidad — {N:,} réplicas')
axes[0].legend(fontsize=9)

# --- FDA ---
sorted_U = np.sort(U) / 1e6
cdf      = np.arange(1, N + 1) / N
p_loss   = np.mean(U < 0)
axes[1].plot(sorted_U, cdf, color='#0D2240', lw=2)
axes[1].axvline(0, color='#C62828', lw=1.5, ls='--', label='U = 0')
axes[1].axhline(p_loss, color='#C62828', lw=1.5, ls=':',
                label=f'P(U < 0) = {p_loss:.2%}')
axes[1].fill_between(sorted_U[sorted_U <= 0], cdf[sorted_U <= 0],
                     alpha=0.25, color='#C62828', label='Zona de pérdida')
axes[1].set_xlabel('Utilidad (M$ COP)'); axes[1].set_ylabel('Probabilidad acumulada')
axes[1].set_title('FDA de la Utilidad Anual'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## Métricas Estadísticas de la Simulación

A partir de las $n$ réplicas calculamos estimadores de los parámetros de interés:

**Media muestral** (estimador de $\mathbb{E}[U]$):
$$\hat{\mu} = \bar{U} = \frac{1}{n}\sum_{i=1}^n u_i$$

**Intervalo de confianza al 95%** para $\mu$ (con $t$-Student):
$$\bar{U} \;\pm\; t_{0.975,\; n-1} \cdot \frac{S}{\sqrt{n}}$$

**Probabilidad de pérdida** (estimador de $P(U < 0)$, variable Bernoulli):
$$\hat{p}_{\text{pérdida}} = \frac{\#\{u_i < 0\}}{n}, \qquad SE = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

<div class="nota">

El **error estándar** $SE = S/\sqrt{n}$ mide la precisión de la estimación de Monte Carlo. Con $n = 10{,}000$ réplicas se logra una precisión alta a bajo costo computacional.

</div>

In [ ]:
from scipy import stats
import numpy as np

# U disponible del kernel (definido en la celda de simulación completa)
n     = len(U)
media = np.mean(U)
s     = np.std(U, ddof=1)
se    = s / np.sqrt(n)
ic    = stats.t.interval(0.95, df=n-1, loc=media, scale=se)
p_loss = np.mean(U < 0)
se_p   = np.sqrt(p_loss * (1 - p_loss) / n)

print("╔══════════════════════════════════════════════════════════╗")
print("║     RESUMEN ESTADÍSTICO — Utilidad Anual (M$ COP)       ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  N réplicas                  :  {n:>10,}              ║")
print(f"║  Media  E[U]                 :  ${media/1e6:>9.3f} M           ║")
print(f"║  Desviación estándar  S      :  ${s/1e6:>9.3f} M           ║")
print(f"║  Error estándar  SE          :  ${se/1e6:>9.5f} M           ║")
print(f"║  IC 95% para μ               :  [${ic[0]/1e6:.3f}M,  ${ic[1]/1e6:.3f}M] ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  P(Pérdida)   P(U < 0)       :  {p_loss:>9.3%}             ║")
print(f"║  SE de P(Pérdida)            :  {se_p:>9.4%}             ║")
print(f"║  P(Ganancia)  P(U > 0)       :  {1-p_loss:>9.3%}             ║")
print(f"║  Mediana                     :  ${np.median(U)/1e6:>9.3f} M           ║")
print(f"║  Percentil 10%               :  ${np.percentile(U,10)/1e6:>9.3f} M           ║")
print(f"║  Percentil 90%               :  ${np.percentile(U,90)/1e6:>9.3f} M           ║")
print("╚══════════════════════════════════════════════════════════╝")

## Métricas de Riesgo: VaR y CVaR

<div class="defn">

**Valor en Riesgo (VaR)** al nivel $\alpha$: umbral tal que la utilidad queda **por debajo** de ese valor con probabilidad $1 - \alpha$.

$$\text{VaR}_\alpha = F_U^{-1}(1 - \alpha) = \text{percentil}_{(1-\alpha) \times 100}(U)$$

**CVaR** (Conditional Value at Risk): utilidad *promedio* en el peor $(1-\alpha)$% de escenarios.

$$\text{CVaR}_\alpha = \mathbb{E}[U \mid U \leq \text{VaR}_\alpha]$$

</div>

**Interpretación para $\alpha = 0.95$:**

| Métrica | Pregunta que responde |
|---|---|
| $\text{VaR}_{0.95}$ | ¿Cuál es el umbral inferior del 5% de peores escenarios? |
| $\text{CVaR}_{0.95}$ | Si ocurre uno de esos peores escenarios, ¿cuánto se pierde en promedio? |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# U disponible del kernel (celda de simulación)
alpha  = 0.95
var95  = np.percentile(U, (1 - alpha) * 100)   # percentil 5 (cola izquierda de U)
cvar95 = U[U <= var95].mean()

fig, ax = plt.subplots(figsize=(10, 4))

# KDE sobre toda la distribución
kde     = stats.gaussian_kde(U / 1e6)
x_range = np.linspace(U.min()/1e6, U.max()/1e6, 500)
ax.plot(x_range, kde(x_range), '#0D2240', lw=2)
ax.fill_between(x_range, kde(x_range), alpha=0.2, color='#1A7A9A', label='Distribución de $U$')

# Colorear la cola izquierda (peor 5%)
x_tail = x_range[x_range <= var95/1e6]
ax.fill_between(x_tail, kde(x_tail), color='#C62828', alpha=0.65, label='Peor 5% de escenarios')

ax.axvline(var95/1e6,  color='#C62828', lw=2.5, ls='--',
           label=f'$\\mathrm{{VaR}}_{{95}}$  = ${var95/1e6:.1f}$ M')
ax.axvline(cvar95/1e6, color='#6A1B9A', lw=2.5, ls='-',
           label=f'$\\mathrm{{CVaR}}_{{95}}$ = ${cvar95/1e6:.1f}$ M')
ax.axvline(0, color='black', lw=1, ls=':', label='U = 0 (equilibrio)')

ax.set_xlabel('Utilidad anual (M$ COP)'); ax.set_ylabel('Densidad')
ax.set_title('Métricas de Riesgo: VaR y CVaR al 95%')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f"\n  VaR₉₅  = ${var95/1e6:.2f} M")
print(f"  → En el 5% de peores escenarios, la utilidad no supera este valor.")
print(f"\n  CVaR₉₅ = ${cvar95/1e6:.2f} M")
print(f"  → Si estamos en ese 5% adverso, la utilidad promedio esperada es ${cvar95/1e6:.2f} M.")

## Análisis de Sensibilidad

**Pregunta:** ¿Cuál variable de entrada tiene mayor impacto sobre la utilidad?

Se mide con la **correlación de rango de Spearman** entre cada entrada y la salida:

$$\rho_s(X_k,\, U) = \text{Corr}\!\left(\text{rango}(X_k),\ \text{rango}(U)\right)$$

| $|\rho_s|$ | Interpretación |
|---|---|
| $> 0.8$ | Influencia muy alta |
| $0.5 – 0.8$ | Influencia alta |
| $0.2 – 0.5$ | Influencia moderada |
| $< 0.2$ | Influencia baja |

El **diagrama de tornado** ordena las variables de mayor a menor influencia absoluta. Indica dónde enfocar esfuerzos de control para reducir el riesgo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Cv, D, U disponibles del kernel (celda de simulación)
rho_cv, _ = spearmanr(Cv, U)
rho_d,  _ = spearmanr(D,  U)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Diagrama de Tornado ---
variables = ['Demanda  D', 'Costo variable  $C_v$']
rhos      = [rho_d, rho_cv]
colors    = ['#1A7A9A' if r > 0 else '#C62828' for r in rhos]
y_pos     = list(range(len(variables)))
axes[0].barh(y_pos, rhos, color=colors, edgecolor='white', height=0.45)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_yticks(y_pos); axes[0].set_yticklabels(variables, fontsize=11)
axes[0].set_xlabel('Correlación de Spearman  ρ')
axes[0].set_title('Diagrama de Tornado — Análisis de Sensibilidad')
axes[0].set_xlim(-1.15, 1.15)
for i, r in enumerate(rhos):
    offset = 0.05 if r > 0 else -0.05
    ha     = 'left' if r > 0 else 'right'
    axes[0].text(r + offset, i, f'{r:.3f}', va='center', ha=ha,
                 fontsize=11, fontweight='bold')

# --- Scatter: utilidad vs demanda ---
axes[1].scatter(D, U/1e6, s=2, alpha=0.2, c='#1A7A9A')
axes[1].set_xlabel('Demanda (unidades/año)')
axes[1].set_ylabel('Utilidad (M$ COP)')
axes[1].set_title(f'Utilidad vs Demanda  (ρ = {rho_d:.3f})')
axes[1].axhline(0, color='#C62828', lw=1.5, ls='--', label='U = 0')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"\n  Variable más influyente: {'Demanda D' if abs(rho_d) > abs(rho_cv) else 'Costo variable Cv'}")
print(f"  → Controlar la demanda tiene mayor impacto en la utilidad.")

## Convergencia: ¿Cuántas Réplicas son Suficientes?

El **error estándar** de la estimación decrece con $\sqrt{n}$:

$$SE(\hat{\mu}) = \frac{S}{\sqrt{n}} \quad\Rightarrow\quad \text{Ancho IC 95\%} \approx \frac{2 \times 1.96 \times S}{\sqrt{n}}$$

**Implicación práctica:**
- Duplicar la precisión requiere **4× más réplicas** (ley del $\sqrt{n}$)
- Para $n$ grande, el IC se estrecha pero hay rendimientos decrecientes

**Criterio de parada:** definir una **tolerancia** $\varepsilon$ (error máximo aceptable) y encontrar $n^*$ tal que:

$$\text{Ancho IC 95\%} \;\leq\; 2\varepsilon \quad\Longleftrightarrow\quad n^* \approx \left(\frac{1.96 \cdot S}{\varepsilon}\right)^2$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros (definidos en la celda de simulación)
P, Cf = 90_000, 60_000_000
cv_min, cv_max = 35_000, 55_000
d_mu, d_sigma  = 2_500, 500

np.random.seed(2024)
N_max  = 20_000
Cv_big = np.random.uniform(cv_min, cv_max, N_max)
D_big  = np.random.normal(d_mu, d_sigma, N_max)
U_big  = (P - Cv_big) * D_big - Cf
mu_ref = np.mean(U_big) / 1e6

ns        = np.unique(np.logspace(1, np.log10(N_max), 300).astype(int))
medias    = np.array([np.mean(U_big[:k]) for k in ns]) / 1e6
ic_widths = np.array([2*1.96*np.std(U_big[:k], ddof=1) / (np.sqrt(k)*1e6) for k in ns])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Convergencia de la media ---
axes[0].semilogx(ns, medias, '#0D2240', lw=1.5, label='$\\hat{\\mu}$ estimado')
axes[0].axhline(mu_ref, color='#C8961E', ls='--', lw=1.5, label=f'Convergencia: ${mu_ref:.2f}M')
axes[0].fill_between(ns, medias - ic_widths/2, medias + ic_widths/2,
                     color='#1A7A9A', alpha=0.2, label='IC 95%')
axes[0].set_xlabel('Número de réplicas $n$')
axes[0].set_ylabel('Utilidad media estimada (M$ COP)')
axes[0].set_title('Convergencia de $E[U]$'); axes[0].legend(fontsize=9)

# --- Reducción del error ---
eps = 0.5
n_needed = next((ns[i] for i, w in enumerate(ic_widths) if w <= eps), None)
axes[1].semilogx(ns, ic_widths, '#C62828', lw=2)
axes[1].axhline(eps, color='#1A7A9A', ls='--', lw=2, label=f'Tolerancia: ancho ≤ ${eps:.1f}M')
if n_needed:
    axes[1].axvline(n_needed, color='#0D2240', ls=':', lw=2,
                    label=f'$n^* \\approx$ {n_needed:,}')
axes[1].set_xlabel('Número de réplicas $n$')
axes[1].set_ylabel('Ancho del IC 95% (M$ COP)')
axes[1].set_title('Reducción del Error de Estimación')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

if n_needed:
    print(f"  Para IC 95% con ancho ≤ ${eps/2:.2f}M → se necesitan aprox. n ≥ {n_needed:,} réplicas.")

## Preguntas de Práctica

Con la simulación del caso de empaques sostenibles, responda:

1. **Probabilidad** — ¿Cuál es $P(U > 80\text{M})$? ¿Y $P(U < -10\text{M})$? Calcúlelas directamente del vector `U`.

2. **Intervalo de confianza** — ¿Cómo cambia el ancho del IC 95% si usa $n = 100$ vs $n = 10{,}000$ réplicas? ¿Cuánto más amplio es?

3. **Punto de equilibrio** — Si $C_v = E[C_v] = 45k$ (fijo), ¿cuál es la demanda mínima $d^*$ para que $U = 0$? Compruébelo con la simulación: calcule $P(D > d^*)$.

4. **Escenario optimista** — Si la empresa negocia con proveedores y reduce el costo variable a $U(30k, 48k)$, ¿cómo cambian $P(\text{pérdida})$ y el $\text{VaR}_{95}$?

5. **Tamaño de muestra** — ¿Cuántas réplicas $n^*$ se necesitan para estimar $P(U < 0)$ con $SE \leq 0.5\%$? Use la fórmula $SE = \sqrt{\hat{p}(1-\hat{p})/n}$.

<div class="nota">

**Pista para la pregunta 5:** despeje $n$ de $SE \leq 0.005$ y use $\hat{p}$ del resultado de la simulación como estimación inicial.

</div>